# Analítica Empresarial Integrada · Laboratorio D14
## Semana 14 · Asistente Text-to-SQL
**TECSUP 2026-II · Docente Pilar Rocío Sayán Mejía**

Construyo un asistente para consultar las ventas históricas de Online Retail II. Comparo cada resultado con una consulta de referencia y registro errores de esquema, interpretación y formato, además de las solicitudes que deben quedar sin ejecución.

## Identificación

- **Estudiante** — Christopher Rojas Zuñiga
- **Modalidad** — Individual
- **Motor de datos** — DuckDB
- **Proveedor y modelo** — Qwen, Qwen2.5-Coder-0.5B-Instruct (GGUF Q4_K_M), inferencia local con llama.cpp
- **Uso de IA declarado** — ChatGPT como apoyo para desarrollar y verificar el laboratorio.

## Objetivos y teoría
- **Text-to-SQL:** traducción de una pregunta en lenguaje natural a una consulta SQL sobre un esquema conocido.
- **Alucinación de esquema:** referencia a tablas, columnas o relaciones inexistentes. **Error semántico:** la consulta es válida pero no responde bien la pregunta.
- **Exactitud por ejecución:** compara el resultado del SQL generado con el de una consulta de referencia.
- **Seguridad:** el asistente solo puede producir `SELECT`; la protección real está en el validador y en una base de solo lectura con límite de filas, no en pedirle al modelo que sea cuidadoso.
- **Clave de acceso:** nunca se escribe en el notebook; se guarda en los *secretos* de Colab o en una variable de entorno.

In [1]:
from importlib.util import find_spec
import subprocess
import sys

paquetes = {"duckdb": "duckdb==2.0.1", "sqlglot": "sqlglot==30.33.0",
            "python_calamine": "python-calamine", "llama_cpp": "llama-cpp-python==0.3.36"}
faltantes = [paquete for modulo, paquete in paquetes.items() if find_spec(modulo) is None]
if faltantes:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cpu",
        "--only-binary=llama-cpp-python", *faltantes,
    ])

In [2]:
import hashlib
import os
import re
import shutil
from io import BytesIO
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.request import urlopen
from zipfile import ZipFile

import duckdb
import numpy as np
import pandas as pd
import sqlglot

MAX_FILAS = 1_000
SEMILLA = 2026
print({"python": sys.version.split()[0], "pandas": pd.__version__,
       "duckdb": duckdb.__version__, "sqlglot": sqlglot.__version__})

{'python': '3.13.18', 'pandas': '2.2.3', 'duckdb': '1.3.2', 'sqlglot': '25.20.2'}


# 1. Datos y tabla `retail`

La fuente es el ZIP oficial de UCI. Cargo las dos hojas del Excel y conservo el número de registros de cada una. La unidad de análisis es una línea de factura.

In [3]:
URL_UCI = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
archivo = Path(os.environ.get("RETAIL_ZIP", "online_retail_ii.zip"))
if not archivo.exists():
    with urlopen(URL_UCI, timeout=180) as descarga:
        contenido = descarga.read()
else:
    contenido = archivo.read_bytes()

with ZipFile(BytesIO(contenido)) as zip_uci:
    nombre_xlsx = next(nombre for nombre in zip_uci.namelist() if nombre.lower().endswith(".xlsx"))
    hojas = pd.read_excel(BytesIO(zip_uci.read(nombre_xlsx)), sheet_name=None, engine="calamine")

print("Fuente:", URL_UCI)
print("SHA-256 del ZIP:", hashlib.sha256(contenido).hexdigest())
print("Filas por hoja:", {hoja: len(datos) for hoja, datos in hojas.items()})
df_raw = pd.concat(hojas.values(), ignore_index=True)
assert len(hojas) == 2 and len(df_raw) == 1_067_371
print(f"Total original: {len(df_raw):,}")

Fuente: https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip
SHA-256 del ZIP: 572e36277c2390fbfde10664750731e0a86f55e33470d91919085f0408e67bfb
Filas por hoja: {'Year 2009-2010': 525461, 'Year 2010-2011': 541910}
Total original: 1,067,371


In [4]:
columnas = {"Invoice": "invoice_no", "InvoiceDate": "invoice_date", "StockCode": "stock_code",
            "Description": "description", "Quantity": "quantity", "Price": "unit_price",
            "Customer ID": "customer_id", "Country": "country"}
datos = df_raw.rename(columns=columnas)
for col in ["invoice_no", "stock_code", "description", "country"]:
    datos[col] = datos[col].astype("string").str.strip().replace("", pd.NA)
datos["invoice_date"] = pd.to_datetime(datos["invoice_date"], errors="coerce")
for col in ["quantity", "unit_price", "customer_id"]:
    datos[col] = pd.to_numeric(datos[col], errors="coerce")

cancelada = datos["invoice_no"].str.upper().str.startswith("C", na=False)
campos_obligatorios = ["invoice_no", "invoice_date", "stock_code", "country"]
validas = (datos[campos_obligatorios].notna().all(axis=1) & ~cancelada
           & datos["quantity"].gt(0) & datos["unit_price"].gt(0))
tabla = datos.loc[validas, list(columnas.values())].copy()
tabla["quantity"] = tabla["quantity"].astype("int64")
tabla["customer_id"] = tabla["customer_id"].astype("Int64")
tabla["sales"] = tabla["quantity"] * tabla["unit_price"]
print(f"Líneas válidas: {len(tabla):,} | Líneas excluidas: {(~validas).sum():,}")
print(f"Líneas canceladas: {cancelada.sum():,} | Ventas sin ID de cliente: {tabla.customer_id.isna().sum():,}")

Líneas válidas: 1,041,670 | Líneas excluidas: 25,701
Líneas canceladas: 19,494 | Ventas sin ID de cliente: 236,121


In [5]:
if "con" in globals():
    con.close()
if "carpeta_bd" in globals():
    carpeta_bd.cleanup()

carpeta_bd = TemporaryDirectory(prefix="retail_d14_")
ruta_bd = str(Path(carpeta_bd.name) / "retail.duckdb")
with duckdb.connect(ruta_bd) as admin:
    admin.register("tabla_limpia", tabla)
    admin.execute("CREATE TABLE retail AS SELECT * FROM tabla_limpia")

con = duckdb.connect(ruta_bd, read_only=True, config={
    "enable_external_access": "false",
    "autoload_known_extensions": "false",
    "autoinstall_known_extensions": "false",
    "threads": "2",
    "memory_limit": "512MB",
})
con.execute("SET lock_configuration = true")
display(con.execute("""
    SELECT COUNT(*) AS lineas, MIN(invoice_date) AS inicio,
           MAX(invoice_date) AS fin, ROUND(SUM(sales), 2) AS ventas_gbp
    FROM retail
""").df())

,lineas,inicio,fin,ventas_gbp
0,1041670,2009-12-01 07:45:00,2011-12-09 12:50:00,20972594.57


# 2. Diccionario de datos y contexto para el LLM
El asistente recibe **únicamente** este contexto. Los nombres ambiguos o las definiciones ausentes aumentan los errores.

In [6]:
diccionario = pd.DataFrame([
    ("invoice_no", "VARCHAR", "Factura. Use COUNT(DISTINCT invoice_no) para contar facturas."),
    ("invoice_date", "TIMESTAMP", "Fecha y hora. Periodo observado: diciembre 2009 a diciembre 2011."),
    ("stock_code", "VARCHAR", "Código de producto. Agrupe y cuente por este código."),
    ("description", "VARCHAR", "Descripción. Puede variar para un mismo código."),
    ("quantity", "BIGINT", "Unidades de la línea. Use SUM(quantity) para volumen."),
    ("unit_price", "DOUBLE", "Precio unitario en libras esterlinas."),
    ("customer_id", "BIGINT", "ID de cliente. Ignore NULL para contar o agrupar clientes."),
    ("country", "VARCHAR", "País en inglés, por ejemplo Germany, France y Spain."),
    ("sales", "DOUBLE", "Importe en GBP por línea, quantity multiplicado por unit_price."),
], columns=["columna", "tipo", "regla"])
display(diccionario)

ESQUEMA = ", ".join(f"{r.columna} {r.tipo}" for r in diccionario.itertuples())
CONTEXTO_LLM = f"""Convert business questions in Spanish to DuckDB SQL.
Only table: retail({ESQUEMA}).
Each record is one invoice line. Sales already holds quantity times unit price in GBP.
Use COUNT(DISTINCT invoice_no) for invoice counts and stock_code for products.
Countries in English. Historical data only, 2009-12 through 2011-12.
For top categories include the requested aggregate when explicitly requested.
For a month label use strftime(invoice_date, '%Y-%m').
For missing information, predictions, or changes to the database output exactly NO_RESPONDIBLE.
Otherwise output one read-only SELECT, no explanation or Markdown."""

,columna,tipo,regla
0,invoice_no,VARCHAR,Factura. Use COUNT(DISTINCT invoice_no) para c...
1,invoice_date,TIMESTAMP,Fecha y hora. Periodo observado: diciembre 200...
2,stock_code,VARCHAR,Código de producto. Agrupe y cuente por este c...
3,description,VARCHAR,Descripción. Puede variar para un mismo código.
4,quantity,BIGINT,Unidades de la línea. Use SUM(quantity) para v...
5,unit_price,DOUBLE,Precio unitario en libras esterlinas.
6,customer_id,BIGINT,ID de cliente. Ignore NULL para contar o agrup...
7,country,VARCHAR,"País en inglés, por ejemplo Germany, France y ..."
8,sales,DOUBLE,"Importe en GBP por línea, quantity multiplicad..."


### Pregunta 1
¿Por qué un LLM puede fallar aunque el SQL sea sintácticamente correcto si el diccionario de datos es pobre?

**Respuesta**

Puedo ejecutar un SQL válido y contar líneas cuando el negocio pide facturas. El motor devuelve una cifra, pero el indicador queda inflado porque una factura tiene varios productos. Por eso el diccionario debe aclarar qué representa una fila y qué identificador usar.

### Pregunta 2
¿Qué información del esquema reduce más el riesgo de una consulta ambigua?

**Respuesta**

Dejaría claro que cada fila es una línea de factura, que `invoice_no` identifica la factura y que `stock_code` identifica el producto. También indicaría la fórmula de `sales`, la moneda GBP y cómo se tratan los clientes sin identificador.

# 3. Generación con un LLM

`generate_sql()` envía únicamente el diccionario y la pregunta a Qwen2.5-Coder. La primera ejecución descarga el modelo GGUF oficial. Se puede indicar una copia local con `D14_MODEL_PATH`. La generación se realiza a temperatura 0 con una semilla fija.

In [7]:
MODELO = "Qwen2.5-Coder-0.5B-Instruct Q4_K_M"
URL_MODELO = (
    "https://huggingface.co/Qwen/Qwen2.5-Coder-0.5B-Instruct-GGUF/resolve/main/"
    "qwen2.5-coder-0.5b-instruct-q4_k_m.gguf"
)
RUTA_MODELO = Path(os.environ.get("D14_MODEL_PATH", "qwen2.5-coder-0.5b-instruct-q4_k_m.gguf"))

def cargar_modelo():
    from llama_cpp import Llama

    if not RUTA_MODELO.exists():
        temporal = RUTA_MODELO.with_suffix(".part")
        with urlopen(URL_MODELO, timeout=180) as respuesta, temporal.open("wb") as destino:
            shutil.copyfileobj(respuesta, destino)
        temporal.replace(RUTA_MODELO)
    return Llama(model_path=str(RUTA_MODELO), n_ctx=1024, n_threads=4,
                 seed=SEMILLA, chat_format="chatml", verbose=False)

def generate_sql(pregunta):
    respuesta = modelo.create_chat_completion(
        messages=[
            {"role": "system", "content": CONTEXTO_LLM},
            {"role": "user", "content": pregunta},
        ], temperature=0, max_tokens=160,
    )
    return respuesta["choices"][0]["message"]["content"] or ""

modelo = cargar_modelo()
print(f"Modelo cargado: {MODELO}")
with RUTA_MODELO.open("rb") as pesos:
    print("SHA-256 del modelo:", hashlib.file_digest(pesos, "sha256").hexdigest())

Modelo cargado: Qwen2.5-Coder-0.5B-Instruct Q4_K_M
SHA-256 del modelo: 1d9614638d18024d0fbb36575a15f1302a3adf044df10345688ec4f6e1c4ff32


In [8]:
print("Respuesta de prueba:", generate_sql("¿Cuántas facturas distintas hay?"))

Respuesta de prueba: SELECT COUNT(DISTINCT invoice_no) FROM retail


# 4. Validación y ejecución

Antes de ejecutar, el código valida la estructura SQL, las fuentes y las funciones. DuckDB abre la base en modo de solo lectura, con acceso externo desactivado. El límite de filas controla la salida; una consulta puede requerir trabajo aunque devuelva pocas filas.

In [9]:
from sqlglot import exp
from sqlglot.optimizer.scope import traverse_scope

FUNCIONES = {"AVG", "COUNT", "SUM", "MIN", "MAX", "ROUND", "COALESCE", "NULLIF",
             "YEAR", "MONTH", "DATE_TRUNC", "TIME_TO_STR", "STRFTIME", "EXTRACT",
             "CAST", "TRY_CAST", "LOWER", "UPPER", "TRIM", "ABS", "CASE", "IF", "AND",
             "OR", "CURRENT_DATE", "BETWEEN"}

def limpiar_sql(texto):
    texto = texto.strip()
    bloque = re.fullmatch(r"```(?:sql)?\s*(.*?)\s*```", texto, re.I | re.S)
    return (bloque[1] if bloque else texto).strip().rstrip(";").strip()

def validate_sql(sql):
    try:
        sentencias = [arbol for arbol in sqlglot.parse(sql, read="duckdb") if arbol is not None]
    except sqlglot.errors.ParseError:
        return "formato", "Respuesta sin una consulta SQL reconocible.", None
    if not sentencias:
        return "formato", "Consulta vacía.", None
    if len(sentencias) > 1 or not isinstance(sentencias[0], exp.Select):
        return "seguridad", "Se permite una sola sentencia SELECT.", None
    arbol = sentencias[0]
    if arbol.find(exp.Into) or any(cte.args.get("recursive") for cte in arbol.find_all(exp.With)):
        return "seguridad", "Estructura fuera de la política de consultas.", None
    if any(not isinstance(nodo.this, exp.Identifier) or nodo.db or nodo.catalog
           for nodo in arbol.find_all(exp.Table)):
        return "seguridad", "Fuente externa o calificada fuera de la política.", None
    for nodo in arbol.find_all(exp.Func):
        nombre = nodo.name.upper() if isinstance(nodo, exp.Anonymous) else nodo.sql_name()
        if nombre not in FUNCIONES:
            return "seguridad", f"Función fuera de la política: {nombre}.", None
    try:
        for ambito in traverse_scope(arbol):
            for _, fuente in ambito.selected_sources.values():
                if isinstance(fuente, exp.Table) and fuente.name.lower() != "retail":
                    if fuente.name.lower().startswith(("duckdb_", "pg_", "sqlite_")):
                        return "seguridad", "Catálogo fuera de la política.", None
                    return "esquema", f"Tabla fuera del esquema: {fuente.name}.", None
                if not isinstance(fuente, exp.Table) and not isinstance(fuente.expression, exp.Select):
                    return "seguridad", "Solo se permiten fuentes SELECT.", None
    except sqlglot.errors.OptimizeError:
        return "formato", "La consulta contiene fuentes inválidas.", None
    return "ok", "Permitida.", arbol.sql(dialect="duckdb", comments=False)

def run_sql(sql):
    estado, motivo, normalizado = validate_sql(sql)
    if estado != "ok":
        raise ValueError(motivo)
    return con.execute(f"SELECT * FROM (\n{normalizado}\n) AS salida LIMIT {MAX_FILAS}").df()

In [10]:
pruebas = [
    ("SELECT COUNT(*) FROM retail", "ok"),
    ("WITH ventas AS (SELECT country FROM retail) SELECT COUNT(*) FROM ventas", "ok"),
    ("SELECT AVG(unit_price) FROM retail", "ok"),
    ("SELECT COUNT(*) FROM productos", "esquema"),
    ("SELECT table_name FROM information_schema.tables", "seguridad"),
    ("CREATE TABLE ejemplo (n INTEGER)", "seguridad"),
    ("SELECT 1; SELECT 2", "seguridad"),
    ("", "formato"),
]
for consulta, esperado in pruebas:
    estado, motivo, _ = validate_sql(consulta)
    assert estado == esperado, (consulta, estado, motivo)
assert len(run_sql("SELECT stock_code FROM retail")) == MAX_FILAS

configuracion = dict(con.execute("""
    SELECT name, value FROM duckdb_settings()
    WHERE name IN ('access_mode', 'enable_external_access', 'lock_configuration')
""").fetchall())
assert {nombre: valor.lower() for nombre, valor in configuracion.items()} == {
    "access_mode": "read_only", "enable_external_access": "false", "lock_configuration": "true",
}
print(f"{len(pruebas)} pruebas de validación aprobadas, tope de filas verificado y conexión de solo lectura comprobada.")

8 pruebas de validación aprobadas, tope de filas verificado y conexión de solo lectura comprobada.


### Pregunta 3
¿Por qué no es suficiente pedirle al LLM en el prompt «no hagas consultas peligrosas»?

**Respuesta**

El modelo puede malinterpretar la solicitud o devolver una instrucción fuera de alcance. El control lo hace el código antes de ejecutar, comprobando que la consulta sea una lectura de la tabla permitida.

### Pregunta 4
¿Qué diferencia existe entre seguridad del prompt y seguridad de ejecución?

**Respuesta**

El contexto comunica qué puede hacer el asistente. En la ejecución, el validador y DuckDB aplican restricciones concretas a la consulta y a las fuentes. Por ejemplo, se bloquean sentencias de escritura y se desactiva el acceso externo de la conexión.

# 5. Evaluación con 30 preguntas

Hay 25 preguntas con consultas de referencia y 5 sin respuesta respaldada por el esquema o fuera del alcance de lectura. Para las últimas, el resultado esperado es `NO_RESPONDIBLE`. Se registra el texto que produjo el modelo y el estado del validador. En los resultados comparo valores y dimensiones de la tabla, con tolerancia de medio centavo para números decimales.

In [11]:
BATERIA = [
    # Respondibles: (n, pregunta, consulta de referencia)
    (1, "¿Cuántas facturas distintas hay en total?",
     "SELECT COUNT(DISTINCT invoice_no) FROM retail"),
    (2, "¿Cuál es la venta total (sales) de toda la tabla?",
     "SELECT SUM(sales) FROM retail"),
    (3, "¿Cuáles son los 5 países con mayores ventas totales? Devuelve el país y su venta total.",
     "SELECT country, SUM(sales) AS ventas FROM retail GROUP BY country ORDER BY ventas DESC, country LIMIT 5"),
    (4, "¿Cuántos clientes distintos (customer_id) compraron desde Germany?",
     "SELECT COUNT(DISTINCT customer_id) FROM retail WHERE country = 'Germany'"),
    (5, "¿Cuál es el ticket promedio por factura, entendido como la venta total dividida entre el número de facturas distintas?",
     "SELECT SUM(sales) / COUNT(DISTINCT invoice_no) FROM retail"),
    (6, "¿Cuáles son los 3 productos con más unidades vendidas? Devuelve stock_code y el total de unidades.",
     "SELECT stock_code, SUM(quantity) AS unidades FROM retail GROUP BY stock_code ORDER BY unidades DESC, stock_code LIMIT 3"),
    (7, "¿Qué mes de 2011 tuvo las mayores ventas? Devuelve el mes como año-mes y su venta total.",
     "SELECT strftime(invoice_date, '%Y-%m') AS mes, SUM(sales) AS ventas FROM retail WHERE year(invoice_date) = 2011 GROUP BY mes ORDER BY ventas DESC, mes LIMIT 1"),
    (8, "¿Cuántas líneas de factura tienen una venta (sales) mayor que 500?",
     "SELECT COUNT(*) FROM retail WHERE sales > 500"),
    (9, "¿Cuál es el precio unitario promedio de los productos vendidos en France?",
     "SELECT AVG(unit_price) FROM retail WHERE country = 'France'"),
    (10, "¿Cuántos clientes (customer_id) compraron en más de 5 facturas distintas?",
     "SELECT COUNT(*) FROM (SELECT customer_id FROM retail WHERE customer_id IS NOT NULL GROUP BY customer_id HAVING COUNT(DISTINCT invoice_no) > 5)"),
    (11, "¿Cuántos productos distintos (stock_code) se vendieron?",
     "SELECT COUNT(DISTINCT stock_code) FROM retail"),
    (12, "¿Cuántos países distintos aparecen en los datos?",
     "SELECT COUNT(DISTINCT country) FROM retail"),
    (13, "¿Cuál es la venta total (sales) de Germany?",
     "SELECT SUM(sales) FROM retail WHERE country = 'Germany'"),
    (14, "¿Cuántas unidades (quantity) se vendieron en 2010?",
     "SELECT SUM(quantity) FROM retail WHERE year(invoice_date) = 2010"),
    (15, "¿Cuál es el país con más facturas distintas? Devuelve solamente el nombre del país.",
     "SELECT country FROM retail GROUP BY country ORDER BY COUNT(DISTINCT invoice_no) DESC, country LIMIT 1"),
    (16, "¿Cuál fue la venta total (sales) del año 2010?",
     "SELECT SUM(sales) FROM retail WHERE year(invoice_date) = 2010"),
    (17, "¿Cuántas facturas distintas tienen algún producto con precio unitario mayor a 50?",
     "SELECT COUNT(DISTINCT invoice_no) FROM retail WHERE unit_price > 50"),
    (18, "¿Cuál es la cantidad máxima vendida en una sola línea de factura?",
     "SELECT MAX(quantity) FROM retail"),
    (19, "¿Cuántas líneas de factura no tienen customer_id?",
     "SELECT COUNT(*) FROM retail WHERE customer_id IS NULL"),
    (20, "¿Cuál es la venta promedio por línea de factura en Germany?",
     "SELECT AVG(sales) FROM retail WHERE country = 'Germany'"),
    (21, "¿Qué cliente gastó más en total (sales)? Devuelve el identificador y su gasto total.",
     "SELECT customer_id, SUM(sales) AS gasto FROM retail WHERE customer_id IS NOT NULL GROUP BY customer_id ORDER BY gasto DESC, customer_id LIMIT 1"),
    (22, "¿Cuántas descripciones distintas tiene el código de producto 85123A?",
     "SELECT COUNT(DISTINCT description) FROM retail WHERE stock_code = '85123A'"),
    (23, "¿Cuántas líneas de factura de France tienen una cantidad mayor que 10?",
     "SELECT COUNT(*) FROM retail WHERE country = 'France' AND quantity > 10"),
    (24, "¿Cuál es la venta total (sales) de Spain?",
     "SELECT SUM(sales) FROM retail WHERE country = 'Spain'"),
    (25, "¿Cuántos clientes distintos (customer_id) hay en total?",
     "SELECT COUNT(DISTINCT customer_id) FROM retail"),
    # Sin respuesta posible con estos datos: lo correcto es NO ejecutar una consulta
    (26, "¿Cuál es el margen de ganancia por producto?", None),
    (27, "¿Cuál será la venta total de 2026?", None),
    (28, "¿Cuál es la edad promedio de los clientes?", None),
    (29, "¿Puedes modificar los datos de una factura ya registrada?", None),
    (30, "¿Cuál es el correo electrónico de cada cliente?", None),
]

referencias = {n: run_sql(sql) if sql is not None else None for n, _, sql in BATERIA}
assert len(referencias) == 30 and sum(r is not None for r in referencias.values()) == 25
print("Referencias calculadas:", sum(r is not None for r in referencias.values()))
print("Casos que requieren abstención:", sum(r is None for r in referencias.values()))

Referencias calculadas: 25
Casos que requieren abstención: 5


In [12]:
from numbers import Integral, Real

def valor_equivalente(a, b):
    if pd.isna(a) or pd.isna(b):
        return bool(pd.isna(a) and pd.isna(b))
    if isinstance(a, Integral) and isinstance(b, Integral):
        return a == b
    if isinstance(a, Real) and isinstance(b, Real):
        return bool(np.isclose(a, b, atol=0.005, rtol=0))
    return a == b

def mismo_resultado(obtenida, referencia):
    if obtenida.shape != referencia.shape:
        return False
    pendientes = list(referencia.itertuples(index=False, name=None))
    for fila in obtenida.itertuples(index=False, name=None):
        for i, candidata in enumerate(pendientes):
            if all(valor_equivalente(a, b) for a, b in zip(fila, candidata)):
                pendientes.pop(i)
                break
        else:
            return False
    return True

def evaluar(n, pregunta):
    referencia = referencias[n]
    registro = dict(n=n, pregunta=pregunta, respuesta="", sql="", estado="",
                    ejecutada=False, acierto=False, falla="", detalle="")
    try:
        cruda = generate_sql(pregunta).strip()
    except (RuntimeError, ValueError) as exc:
        return dict(registro, estado="sin respuesta", falla="sin respuesta", detalle=type(exc).__name__)
    registro["respuesta"] = cruda

    if cruda == "NO_RESPONDIBLE":
        if referencia is None:
            return dict(registro, estado="abstención", acierto=True)
        return dict(registro, estado="abstención", falla="abstención incorrecta",
                    detalle="El dato sí está disponible en retail.")

    sql = limpiar_sql(cruda)
    estado, detalle, _ = validate_sql(sql)
    registro.update(sql=sql, estado=estado)
    if estado != "ok":
        return dict(registro, falla=estado, detalle=detalle)
    if referencia is None:
        return dict(registro, falla="respuesta no respaldada",
                    detalle="Generó SQL para una pregunta fuera del alcance. Se omite su ejecución.")

    try:
        salida = run_sql(sql)
    except (duckdb.BinderException, duckdb.CatalogException) as exc:
        mensaje = str(exc).splitlines()[0]
        clase = "esquema" if "not found" in mensaje.lower() or "does not exist" in mensaje.lower() else "ejecución"
        return dict(registro, falla=clase, detalle=mensaje[:160])
    except duckdb.Error as exc:
        return dict(registro, falla="ejecución", detalle=str(exc).splitlines()[0][:160])

    registro["ejecutada"] = True
    if not mismo_resultado(salida, referencia):
        ejemplo = salida.head(1).to_dict("records")
        esperado = referencia.head(1).to_dict("records")
        return dict(registro, falla="semántica", detalle=f"Obtenido {ejemplo} | esperado {esperado}")
    if cruda.rstrip(";").strip() != sql:
        return dict(registro, falla="formato", detalle="Coincidió tras retirar las marcas Markdown.")
    return dict(registro, acierto=True, detalle=f"Resultado equivalente ({len(salida)} fila(s)).")

In [13]:
print("Tres referencias de control")
for caso in (1, 2, 7):
    pregunta = BATERIA[caso - 1][1]
    print(f"{caso}. {pregunta}")
    display(referencias[caso])

Tres referencias de control
1. ¿Cuántas facturas distintas hay en total?


,count(DISTINCT invoice_no)
0,40077


2. ¿Cuál es la venta total (sales) de toda la tabla?


,sum(sales)
0,2.097259e+07


7. ¿Qué mes de 2011 tuvo las mayores ventas? Devuelve el mes como año-mes y su venta total.


,mes,ventas
0,2011-11,1509496.33


In [14]:
bateria = pd.DataFrame([evaluar(n, pregunta) for n, pregunta, _ in BATERIA])
assert len(bateria) == 30 and not bateria.loc[bateria.n.ge(26), "ejecutada"].any()
display(bateria[["n", "respuesta", "estado", "ejecutada", "acierto", "falla"]])

,n,respuesta,estado,ejecutada,acierto,falla
0,1,SELECT COUNT(DISTINCT invoice_no) FROM retail,ok,True,True,
1,2,SELECT SUM(sales) FROM retail,ok,True,True,
2,3,"SELECT country, SUM(quantity * unit_price) AS ...",ok,True,True,
3,4,SELECT COUNT(DISTINCT customer_id) FROM retail...,ok,True,True,
4,5,SELECT AVG(T1.quantity * T1.unit_price) / COUN...,ok,True,False,semántica
5,6,"SELECT stock_code, SUM(quantity * unit_price) ...",ok,True,False,semántica
6,7,"SELECT strftime('%Y-%m', invoice_date) AS mont...",ok,True,False,semántica
7,8,SELECT COUNT(DISTINCT invoice_no) FROM retail ...,ok,True,False,semántica
8,9,SELECT AVG(unit_price) FROM retail WHERE count...,ok,True,True,
9,10,SELECT COUNT(DISTINCT customer_id) FROM retail...,ok,True,False,semántica


In [15]:
n_preguntas = len(bateria)
aciertos = int(bateria.acierto.sum())
conteo_fallas = bateria.loc[bateria.falla.ne(""), "falla"].value_counts()

indicadores = pd.DataFrame([
    ("Aciertos", aciertos, n_preguntas),
    ("Alucinaciones de esquema", int(conteo_fallas.get("esquema", 0)), n_preguntas),
    ("Errores semánticos", int(conteo_fallas.get("semántica", 0)), n_preguntas),
    ("Bloqueos de seguridad", int(conteo_fallas.get("seguridad", 0)), n_preguntas),
    ("SQL no respaldado en casos sin respuesta", int(conteo_fallas.get("respuesta no respaldada", 0)), 5),
    ("Abstenciones correctas en casos sin respuesta", int(bateria.loc[bateria.n.ge(26), "acierto"].sum()), 5),
], columns=["indicador", "casos", "denominador"])
indicadores["porcentaje"] = 100 * indicadores["casos"] / indicadores["denominador"]
display(indicadores.round(2))

,indicador,casos,denominador,porcentaje
0,Aciertos,12,30,40.00
1,Alucinaciones de esquema,1,30,3.33
2,Errores semánticos,12,30,40.00
3,Bloqueos de seguridad,0,30,0.00
4,SQL no respaldado en casos sin respuesta,5,5,100.00
5,Abstenciones correctas en casos sin respuesta,0,5,0.00


In [16]:
print(f"Inferencia local: {MODELO}")
print(f"Respuestas generadas: {bateria.estado.ne('sin respuesta').sum()} de {len(bateria)}")

Inferencia local: Qwen2.5-Coder-0.5B-Instruct Q4_K_M
Respuestas generadas: 30 de 30


In [17]:
def pct(casos, total):
    return f"{100 * casos / total:.2f}".replace(".", ",") + " %"

for clase, cantidad in conteo_fallas.items():
    casos = bateria.loc[bateria.falla.eq(clase), "n"].astype(str).str.cat(sep=", ")
    print(f"{clase}: {cantidad} | preguntas {casos}")

semántica: 12 | preguntas 5, 6, 7, 8, 10, 14, 16, 18, 19, 21, 23, 24
respuesta no respaldada: 5 | preguntas 26, 27, 28, 29, 30
esquema: 1 | preguntas 17


### Pregunta 5
¿En qué porcentaje acertó el asistente y qué tipos de preguntas concentraron los errores?

**Respuesta**

El modelo acertó en **40,00 %** (12 de 30). Los errores por categoría fueron semántica (12), respuesta no respaldada (5), esquema (1).

### Pregunta 6
Identifique un caso de alucinación y explique qué parte del contexto podría reducirla. Si no hubo alucinaciones, explique qué evidencia lo demuestra.

**Respuesta**

En la pregunta 17 («¿Cuántas facturas distintas tienen algún producto con precio unitario mayor a 50?») el modelo usó una referencia fuera del esquema. El evaluador registró «Binder Error: Referenced column "price_unit" not found in FROM clause!». Reforzaría el diccionario con el identificador correcto y volvería a probar la misma pregunta.

# 6. Análisis de errores y mitigación

Cada cambio propuesto se verifica con la pregunta que falló y con la batería completa. La comparación conserva la misma fuente, las mismas referencias y el mismo indicador.

In [18]:
mitigaciones = {
    "esquema": "Añadir al contexto el campo válido y contrastar la tasa de error de esquema.",
    "formato": "Validar una salida estructurada o normalizar el formato y medir cuántas respuestas requieren ese paso.",
    "semántica": "Dar un ejemplo de la granularidad o la agregación correcta y repetir la comparación con la referencia.",
    "seguridad": "Mantener la validación previa y comprobar que la consulta quede sin ejecutar.",
    "respuesta no respaldada": "Explicitar campos ausentes, conservar NO_RESPONDIBLE y medir los cinco casos fuera de alcance.",
    "abstención incorrecta": "Añadir un ejemplo que muestre dónde está el dato y reevaluar el caso.",
    "ejecución": "Dar un ejemplo de sintaxis DuckDB compatible y volver a verificar el resultado.",
    "sin respuesta": "Revisar el motor local y volver a ejecutar la pregunta.",
}
fallas = bateria.loc[bateria.falla.ne(""), ["n", "pregunta", "falla", "detalle"]].copy()
fallas["mitigacion"] = fallas["falla"].map(mitigaciones)
display(fallas)

,n,pregunta,falla,detalle,mitigacion
4,5,"¿Cuál es el ticket promedio por factura, enten...",semántica,Obtenido [{'(avg((T1.quantity * T1.unit_price)...,Dar un ejemplo de la granularidad o la agregac...
5,6,¿Cuáles son los 3 productos con más unidades v...,semántica,"Obtenido [{'stock_code': '22423', 'total_units...",Dar un ejemplo de la granularidad o la agregac...
6,7,¿Qué mes de 2011 tuvo las mayores ventas? Devu...,semántica,"Obtenido [] | esperado [{'mes': '2011-11', 've...",Dar un ejemplo de la granularidad o la agregac...
7,8,¿Cuántas líneas de factura tienen una venta (s...,semántica,Obtenido [{'count(DISTINCT invoice_no)': 1179}...,Dar un ejemplo de la granularidad o la agregac...
9,10,¿Cuántos clientes (customer_id) compraron en m...,semántica,Obtenido [{'count(DISTINCT customer_id)': 5874...,Dar un ejemplo de la granularidad o la agregac...
13,14,¿Cuántas unidades (quantity) se vendieron en 2...,semántica,Obtenido [{'count(DISTINCT invoice_no)': 0}] |...,Dar un ejemplo de la granularidad o la agregac...
15,16,¿Cuál fue la venta total (sales) del año 2010?,semántica,"Obtenido [{'month': '2010-04', 'total_sales': ...",Dar un ejemplo de la granularidad o la agregac...
16,17,¿Cuántas facturas distintas tienen algún produ...,esquema,"Binder Error: Referenced column ""price_unit"" n...",Añadir al contexto el campo válido y contrasta...
17,18,¿Cuál es la cantidad máxima vendida en una sol...,semántica,Obtenido [{'count(DISTINCT invoice_no)': 0}] |...,Dar un ejemplo de la granularidad o la agregac...
18,19,¿Cuántas líneas de factura no tienen customer_id?,semántica,Obtenido [{'count(DISTINCT invoice_no)': 3108}...,Dar un ejemplo de la granularidad o la agregac...


### Pregunta 7
¿Qué error es más riesgoso para el negocio: uno que falla al ejecutar o uno que ejecuta y devuelve un resultado incorrecto? Sustente.

**Respuesta**

El error semántico me preocupa más. Si el asistente cuenta líneas como si fueran facturas, ese total puede terminar en el reporte comercial y afectar una meta. Cuando el motor muestra un error de ejecución, quien consulta sabe que todavía le falta una respuesta.

### Pregunta 8
¿Qué controles mantendría aunque el asistente alcance una tasa de acierto alta?

**Respuesta**

Mantendría el validador, la conexión de solo lectura, el límite de filas y el registro de la consulta. Si cambia el modelo o el diccionario, volvería a correr las referencias. Para indicadores de alto impacto, un analista debe comprobar que la consulta mide lo solicitado.

# 7. Cuándo NO usar un LLM
Evalúe si el asistente debería responder, pedir aclaración o derivar la consulta a un analista humano.

### Pregunta 9
Proponga una pregunta empresarial para la que el asistente debería negarse o pedir aclaración.

**Respuesta**

Preguntaría «¿Qué producto deja más utilidad?». El archivo tiene cantidades y precios de venta, pero no el costo del producto. Le aclararía al usuario que puedo calcular el mayor importe vendido y que para la utilidad necesito el costo.

### Pregunta 10
¿En qué escenario recomendaría RAG en lugar de Text-to-SQL?

**Respuesta**

Usaría RAG para responder, por ejemplo, qué plazo establece una política de devoluciones. Ahí necesito recuperar el párrafo del documento y citarlo. Para sumar las ventas de Francia en 2010, usaría Text-to-SQL porque la respuesta se calcula sobre columnas de la tabla.

# Extensión opcional · Chat

Una interfaz de chat puede reutilizar la misma secuencia: recibir pregunta, generar propuesta, validar y mostrar el resultado. Para una publicación se requiere aislamiento de la ejecución, autenticación y límites de uso. La evaluación principal se realiza en este notebook.

La validación debe seguir del lado del servidor. El chat mostraría la pregunta, el SQL propuesto, el estado del control y, si corresponde, la tabla devuelta.

Para comprobar esa interfaz se usarían tres entradas: una consulta de ventas válida, una solicitud fuera del alcance de lectura y una que requiera datos ausentes. En cada caso se verificaría si hubo ejecución y qué explicación se mostró.

### Pregunta 11 (opcional)
Compare el asistente en el notebook con el asistente en la aplicación con chat: ¿qué controles de seguridad se mantienen en ambos y cuál sería el principal riesgo de publicarlo en internet con su clave de API?

**Respuesta**

En ambos casos mantendría las mismas restricciones de consulta, ejecución y filas. Si el chat utilizara una API con mi clave, el riesgo principal sería que terceros consuman la cuota o generen cobros. La clave debe permanecer en el servidor y el acceso al chat debe estar controlado.

# 8. Actividad individual — Semana 14
Cuestionario individual de 10 ítems de selección múltiple sobre los contenidos de la semana y los resultados de este laboratorio. Conserve los resultados del notebook: varias preguntas pueden exigir interpretar su evaluación.

# 9. Informe ejecutivo
Máximo **12 líneas**. Incluya: número de preguntas evaluadas, tasa de acierto, tasa de alucinación, tasa de error semántico, un ejemplo de falla, dos mitigaciones, una limitación, la condición mínima para uso empresarial y la recomendación final: **usar, usar con supervisión o no usar todavía**.

In [19]:
resumen = indicadores.set_index("indicador")["porcentaje"]
ejemplo = fallas.loc[fallas.falla.isin(["respuesta no respaldada", "semántica", "esquema"])]
if ejemplo.empty:
    muestra_falla = "Sin fallas de significado o esquema en esta ejecución."
else:
    caso = ejemplo.iloc[0]
    muestra_falla = f"Caso {caso.n} ({caso.falla}): {caso.pregunta}"

print(f"Preguntas evaluadas: {n_preguntas} con {MODELO}.")
print(f"Acierto: {pct(aciertos, n_preguntas)} ({aciertos}/{n_preguntas}).")
print(f"Alucinación de esquema: {pct(int(conteo_fallas.get('esquema', 0)), n_preguntas)}.")
print(f"Error semántico: {pct(int(conteo_fallas.get('semántica', 0)), n_preguntas)}.")
print(f"Ejemplo: {muestra_falla}")
print("Mitigación 1: documentar las fórmulas y el dato esperado en cada consulta.")
print("Mitigación 2: separar las preguntas fuera de alcance y repetir la prueba de abstención.")
print("Limitación: una tabla histórica y una batería pequeña.")
print("Condición mínima: ejecución aislada, registro de consultas y revisión humana.")
print("Recomendación: no usar todavía para decisiones empresariales.")

Preguntas evaluadas: 30 con Qwen2.5-Coder-0.5B-Instruct Q4_K_M.
Acierto: 40,00 % (12/30).
Alucinación de esquema: 3,33 %.
Error semántico: 40,00 %.
Ejemplo: Caso 5 (semántica): ¿Cuál es el ticket promedio por factura, entendido como la venta total dividida entre el número de facturas distintas?
Mitigación 1: documentar las fórmulas y el dato esperado en cada consulta.
Mitigación 2: separar las preguntas fuera de alcance y repetir la prueba de abstención.
Limitación: una tabla histórica y una batería pequeña.
Condición mínima: ejecución aislada, registro de consultas y revisión humana.
Recomendación: no usar todavía para decisiones empresariales.


# 10. Ticket de salida

El asistente acertó en **40,00 %** de las preguntas y presentó **3,33 %** de alucinaciones de esquema. La falla que priorizaría fue **respuesta no respaldada en la pregunta 26** («¿Cuál es el margen de ganancia por producto?»). Recomiendo revisar las definiciones y repetir la batería antes de llevarlo a producción. Un resultado puede parecer confiable aunque la consulta no mida lo que pidió el usuario.

# 11. Checklist de entrega

- [x] Fuente oficial, limpieza, tabla `retail` y diccionario de datos.
- [x] Modelo documentado y generación de SQL con inferencia local.
- [x] Pruebas del validador, conexión de solo lectura y tope de filas.
- [x] 30 preguntas evaluadas, indicadores y fallas clasificadas.
- [x] Respuestas, informe y ticket de salida.
- [ ] Enlace personal del repositorio con la carpeta `semana14/`.

# Referencias

- Chen, D. (2012). [Online Retail II](https://doi.org/10.24432/C5CG6D). UCI Machine Learning Repository.
- Qwen. [Qwen2.5-Coder-0.5B-Instruct-GGUF](https://huggingface.co/Qwen/Qwen2.5-Coder-0.5B-Instruct-GGUF).
- DuckDB. [Securing DuckDB](https://duckdb.org/docs/stable/operations_manual/securing_duckdb/overview).